## Welcome to the 5S-TES Workbench!

This Jupyter notebook walks you through steps to use the 5S-TES Workbench to interact with the Five Safes TES stack.

By the end of this notebook, you will have built a Darwin EU analysis, a TES task, submitted it to the TES server, fetched the outputs and visualised them.

This notebook has some "cells" which is a group of executable Python code. To run the code/cell, you can simply click on the cell and press the play button or pressing `Shift+Enter`.

Before each cell, we will provide a description of the task and the expected outputs. The full guide on how to run Workbench is available [here](https://docs.federated-analytics.ac.uk/tutorials/submit-to-5s-tes/workbench).

Let's get started!

### Cell 1: Initialise the Workbench and pass the configuration

Run the following cell to initialise the Workbench and pass the configuration to it.

Configurations explanation:

- `project`: The name of the project you are working on, e.g. "OHDSIDemo".
- `tres`: The TREs you want to run the analysis on, e.g. "Nottingham TRE 01" and "Nottingham TRE 02".
- `tes_base_url`: The base URL of the TES server. For 5S-TES, this is the Submission layer API endpoint.
- `keycloak_url`: The URL of the Keycloak server.
- `client_id` and `client_secret`: The client ID and client secret for the OIDC Keycloak server.
- `username` and `password`: Your username and password for 5S-TES Submission layer.

**Note**: 
- Apart from username and password, the configuration is pre-filled and should not be changed, unless you are submitting to a different TES server, for example.
- To test the connection to 5S-TES, we included the code to fetch the Hello World output. After running the cell, if you can see the file `stdout.txt` under the `output/Nottingham TRE 01/1857/` directory next to the notebook on the left-side panel, it means the connection is successful.


In [1]:
from ui import IncidencePrevalenceForm, WorkbenchForm

This form handles initialisation of the workbench for you; you only need to enter your user name and Password

In [2]:
wb_form = WorkbenchForm()

wb_form.display()

Run the next cell to check you're connecting OK.
If it doesn't work, you'll get a big scary error message.
Don't worry, just check your credentials!

In [3]:
wb_form.check_connection()

INFO | Template registered: 'hello_world'
INFO | Template registered: 'custom'
INFO | Template registered: 'simple_sql'
INFO | Template registered: 'bunny'
INFO | Template registered: 'analysis'
INFO | Validation successful
INFO | Config: project='OHDSIDemo' tes_base_url='https://api.5s-tes.federated-research.com/' tres=['Nottingham TRE 01', 'Nottingham TRE 02']
INFO | Auth mode: AuthMode.CREDENTIALS
INFO | Fetching Keycloak ID token for STS exchange...
INFO | Requesting Keycloak token from https://drs-core-identity.azurewebsites.net/realms/Dare-Control/protocol/openid-connect/token
INFO | Keycloak token fetched successfully
INFO | Exchanging bearer token for MinIO credentials via STS (https://api.s3.5s-tes.federated-research.com)
INFO | MinIO client initialized (endpoint=https://api.s3.5s-tes.federated-research.com, secure=True)
INFO | Child task info: 1857, TRE: Nottingham TRE 01, status: Completed
INFO | Found 1 result object(s) for task 1857
INFO | Downloading result object: 1857/s

### Cell 2: Build your analysis, TES task and submit it

This notebook will run the [Incidence Prevalence analysis](https://darwin-eu.github.io/IncidencePrevalence/) from [Darwin EU](https://www.darwin-eu.org/index.php) on the [OMOP](https://ohdsi.github.io/CommonDataModel/) database of two TREs (Nottingham TRE 01 and Nottingham TRE 02).

To do this, first, we need to define the concept set:

A concept set is a named list of unique numbers called concept IDs that together define the group of patients you want to study. For example, because the concept ID for skin cancer (or neoplasm) is 139750, I could define the concept set for skin cancer as `{"neoplasm": [139750]}`. For diabetes, the concept set could be `{"diabetes": [4131907,4220821]}`.

To build a concept set for yourself, 
1. Go to https://athena.ohdsi.org
2. Search for your condition (e.g., "skin cancer")
3. Filter by Standard concepts and Condition domain
4. Copy the number(s) in the Concept ID column
5. Put the number(s) in the square brackets, e.g. [139750, 609278, 4133025], and name the concept set, e.g. "skin cancer"

Secondly, we need to put the following information:
- Denominator cohort and cohort name: put a name for the denominator cohort and cohort name in `snake_case`, e.g. "skin_cancer_test_denominator" and "skin_cancer_test"
- Your name, e.g. "John Smith"

Finally, run the cell to build the TES task and submit it to the TES server.

**Note**: 
- Run the cell ONCE, when you are ready. Because each time you run the cell, it will build a new TES task and submit it to the TES server.
- If you submit the TES task successfully, you will see the ID of it in the cell's output, but not the analysis outputs. The TES task needs to be processed by the TRE, TES executor and Egress layer, before the outputs are available and fetched in cell 3.
- String texts, e.g., cohort name, concept set name, etc., should be put inside the double quotes and in `snake_case`, e.g. "skin_cancer_test"


In [4]:
task_form = IncidencePrevalenceForm()
task_form.display()

In [6]:
# Build the TES task - should not be changed
wb = wb_form.validate()

wb.build_tes.custom(
    name=f"OHDSI - {task_form.researcher_name}",
    description="Darwin EU - Incidence prevalence demo analysis",
    executors=task_form.render_executor(),
    outputs=[
        {
            "name": "Analysis Results Location",
            "description": "Analysis Results Location",
            "url": "s3://",
            "path": "/outputs",
            "type": "DIRECTORY",
        }
    ],
)

# Submit the TES task - should not be changed
wb.submit()

INFO | Template registered: 'hello_world'
INFO | Template registered: 'custom'
INFO | Template registered: 'simple_sql'
INFO | Template registered: 'bunny'
INFO | Template registered: 'analysis'
INFO | Validation successful
INFO | Config: project='OHDSIDemo' tes_base_url='https://api.5s-tes.federated-research.com/' tres=['Nottingham TRE 01', 'Nottingham TRE 02']
INFO | Auth mode: AuthMode.CREDENTIALS
INFO | Building TES task from template: 'custom'
INFO | Resolving template: 'custom'
INFO | TES Task built successfully
INFO | TES payload:
{
   "name": "OHDSI - John Snow",
   "description": "Darwin EU - Incidence prevalence demo analysis",
   "outputs": [
      {
         "url": "s3://",
         "path": "/outputs",
         "type": "DIRECTORY",
         "name": "Analysis Results Location",
         "description": "Analysis Results Location"
      }
   ],
   "executors": [
      {
         "image": "ghcr.io/health-informatics-uon/omop-r-tools:sha-9aa526d",
         "command": [
         

'3056'

### Cell 3: Retrieve the outputs

Once the outputs are approved by TRE's Egress officer, you can retrieve the outputs by running the following cell.

By default, without any paramters, function `fetch_outputs` will try fetching the outputs related to the `latest task`, which you just submitted in cell 2, from `all targeted TREs` (i.e., Nottingham TRE 01 and Nottingham TRE 02, which were set in cell 1) and save them in the `output` directory next to this notebook.

You can also fetch outputs from a specific task by passing the `task_id` parameter to the function. For example, to fetch the outputs from task with `task_id = 1823`, you can run: `wb.fetch_outputs(task_id=1823)`


In [ ]:
wb.fetch_outputs()

### Cell 4: Visualise the outputs

This is a WIP.
